# GTEx model building with PLIER

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using PLIER. It automates downloading and preprocessing the GTEx matrix, creates a Filebacked Big Matrix (FBM), computes an SVD to estimate the model dimension, prepares pathway priors, runs PLIER, and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

## Load libraries

In [1]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx PLIER analysis started at:", format(start_time), "\n")

GTEx PLIER analysis started at: 2026-03-30 17:22:02 


In [2]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(123)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [3]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

## Input

In [4]:
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))
CLAMP_K_gtex <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))
gtex_genes <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))

# Settings

In [5]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Prepare pathway priors

In [7]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

# prefix each gene-set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways

Inverting...

done



# PLIER

Run PLIER with the same inputs

In [8]:
CLAMP_K_gtex

[1] 578
attr(,"limit")
[1] 1.530897

In [9]:
gtex_plier = PLIER::PLIER(
    gtex_fbm_filt[], 
    as.matrix(gtex_matched), 
    svdres = gtex_svdRes, 
    Chat = as.matrix(gtex_chatObj), 
    doCrossval = TRUE, 
    k = CLAMP_K_gtex
  )

colnames(gtex_plier$Z) <- paste0('LV', seq_len(ncol(gtex_plier$Z)))
head(gtex_plier$Z)
dim(gtex_plier$Z)

gtex_plier$summary <- gtex_plier$summary %>%
    dplyr::rename(LV = `LV index`)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

colnames(gtex_plier$B) <- samples

saveRDS(gtex_plier, file = file.path(output_data_dir, "PLIER.rds"))

model_dir <- file.path(output_data_dir, "PLIER")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_plier$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- gtex_plier$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- gtex_plier$summary
write.csv(summary, file.path(model_dir, "summary.csv"))

Removing 0 pathways with too few genes



[1] 117.0481
[1] "L2 is set to 117.048092647883"
[1] "L1 is set to 58.5240463239414"


errorY (SVD based:best possible) = 0.6867

New L3 is 0.00012340980408668

New L3 is 7.48518298877006e-05

New L3 is 6.60565080286848e-05

New L3 is 6.60565080286848e-05

New L3 is 6.60565080286848e-05

New L3 is 6.60565080286848e-05

New L3 is 6.60565080286848e-05

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

New L3 is 6.60565080286848e-05

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

converged at  iteration 162 Bdiff is not decreasing

There are 251  LVs with AUC>0.70



,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV569,LV570,LV571,LV572,LV573,LV574,LV575,LV576,LV577,LV578
WASH7P,0.05522751,0.00000000,0.11329158,0.02392497,0.00000000,0.0000000,0.07198731,0.0600441,0.04628241,0.00000000,⋯,0.00000000,0.08287723,0.00000000,0.000000000,0.00000000,0.0007006171,0.03019083,0.02283949,0.00000000,0.003942853
RP11-34P13.15,0.01136521,0.25553502,0.24927735,0.00000000,0.00000000,0.1934606,0.00000000,0.0000000,0.00000000,0.00000000,⋯,0.00000000,0.02393050,0.00000000,0.035288584,0.07699515,0.0249110082,0.00000000,0.05779582,0.00000000,0.000000000
RP11-34P13.16,0.01596191,0.26625383,0.18263380,0.00000000,0.00000000,0.2092903,0.00000000,0.0000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.002261421,0.07693474,0.0101908712,0.00000000,0.02528293,0.00000000,0.011118414
RP11-34P13.18,0.00000000,0.19270114,0.13219478,0.00000000,0.06309370,0.0000000,0.00000000,0.0956906,0.00000000,0.05803359,⋯,0.06672439,0.00000000,0.06691029,0.000000000,0.11492227,0.1691396152,0.00000000,0.00000000,0.00000000,0.000000000
AP006222.2,0.00000000,0.00000000,0.46055646,0.00000000,0.06511794,0.0000000,0.00000000,0.1902355,0.00000000,0.02471497,⋯,0.00000000,0.00000000,0.00000000,0.017613129,0.15109785,0.0935431501,0.00000000,0.17967927,0.01039061,0.000000000
MTND1P23,0.14839772,0.01820273,0.01406149,0.04313769,0.00000000,0.0000000,0.00000000,0.0000000,0.25180513,0.00000000,⋯,0.00000000,0.13055743,0.00000000,0.000000000,0.00000000,0.0000000000,0.00000000,0.14720109,0.00000000,0.000000000


[1] 21613   578